<a href="https://colab.research.google.com/github/LarshVakil/Pokemon-Classification-and-Regression/blob/main/PokemonClassificationRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

#Importing from kagglehub rather than downloading csv and putting in files
import kagglehub
import os
path = kagglehub.dataset_download("abcsds/pokemon")

100%|██████████| 14.9k/14.9k [00:00<00:00, 9.75MB/s]

Extracting files...


In [ ]:
#Loaded dataset
csv_path = os.path.join(path, "Pokemon.csv")
df = pd.read_csv(csv_path)
df.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


In [ ]:
#EDA

print(df.shape)


print(df.describe())

# We see even though 800 enteries only 721 indexes because Charizard , mega charizard X , mega chrizard Y all 3 have index 6
print(df.isnull().sum())

# 386 pokemon are not dual types so second type , type 2 is NaN lets use fillna to clean it

df['Type 2'] = df['Type 2'].fillna('None')

#test
#print(df.isnull().sum())

#Checking is all pokrmon BST adds up

count = 0
for i in range(len(df)):
  if df.loc[i, 'Total'] == df.loc[i, 'HP'] + df.loc[i, 'Attack'] + df.loc[i, 'Defense'] + df.loc[i, 'Sp. Atk'] + df.loc[i, 'Sp. Def'] + df.loc[i, 'Speed']:
    pass
  else:
    count+=1
print(count)

#Checking total legy

total_legy = df['Legendary'].sum()

print(total_legy)



(800, 13)
                #      Total          HP      Attack     Defense     Sp. Atk  \
count  800.000000  800.00000  800.000000  800.000000  800.000000  800.000000   
mean   362.813750  435.10250   69.258750   79.001250   73.842500   72.820000   
std    208.343798  119.96304   25.534669   32.457366   31.183501   32.722294   
min      1.000000  180.00000    1.000000    5.000000    5.000000   10.000000   
25%    184.750000  330.00000   50.000000   55.000000   50.000000   49.750000   
50%    364.500000  450.00000   65.000000   75.000000   70.000000   65.000000   
75%    539.250000  515.00000   80.000000  100.000000   90.000000   95.000000   
max    721.000000  780.00000  255.000000  190.000000  230.000000  194.000000   

          Sp. Def       Speed  Generation  
count  800.000000  800.000000   800.00000  
mean    71.902500   68.277500     3.32375  
std     27.828916   29.060474     1.66129  
min     20.000000    5.000000     1.00000  
25%     50.000000   45.000000     2.00000  
50%  

In [ ]:
# Feature encoding (One Hot)

ohe = OneHotEncoder(handle_unknown='ignore',sparse_output= False).set_output(transform='pandas')
df2 = ohe.fit_transform(df[['Legendary','Type 1','Type 2']])
df2.head()

df = pd.concat([df,df2],axis=1).drop(['Legendary','Type 1','Type 2'],axis=1)
df.head()

,#,Name,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,...,Type 2_Grass,Type 2_Ground,Type 2_Ice,Type 2_None,Type 2_Normal,Type 2_Poison,Type 2_Psychic,Type 2_Rock,Type 2_Steel,Type 2_Water
0,1,Bulbasaur,318,45,49,49,65,65,45,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,Ivysaur,405,60,62,63,80,80,60,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,Venusaur,525,80,82,83,100,100,80,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,3,VenusaurMega Venusaur,625,80,100,123,122,120,80,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,4,Charmander,309,39,52,43,60,50,65,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#Testing , Training with random forest regressor


#---------------------Regression Task ------------------------------------------
#Split into X AND y


# Adding total decreases the mse and r2 but have dropped it intentionally
X = df.drop(columns=['#', 'Name', 'HP','Total'])
y = df['HP']

# Train test split 80 20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

pipe = Pipeline([
    ('model', RandomForestRegressor())
])


#Training
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

#Accuracy metrics as mentioned (Mean Squared Error (MSE) and R-Squared metrics, respectively)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'mse: {mse}')
print(f'r2: {r2}')

Training Data Shape: (640, 123)
Testing Data Shape: (160, 123)
mse: 224.41860312499998
r2: 0.4596053797571049


In [ ]:
#Classificaiton of legendary pokemon